# Pizza Vision: synthetic data seed\n\nPopulates dummy tables for the **Pizza Vision** AppKit app demo.\nTables are written under `${catalog}.${schema}` with sensible defaults\n(`reggie_pierce_7405614800873570.pizza_vision`). Re-run idempotently: the script drops and recreates\neach table before inserting fresh data.\n\nTables created:\n\n- `stores`         (one row per location)\n- `devices`        (temperature sensors, status)\n- `device_readings`(historical temperature / humidity samples)\n- `camera_status`  (online/offline samples per camera, last 24h)\n- `detections`     (CV bounding boxes: pizza, vehicle, person, truck, package)\n- `license_plates` (drive-through plate captures)\n- `inventory`      (pizza stock + truck parking, last 12h + projection)\n- `alerts`         (Jolt rule events, last 7 days)

In [ ]:
dbutils.widgets.text("catalog", "reggie_pierce_7405614800873570")
dbutils.widgets.text("schema", "pizza_vision")
dbutils.widgets.text("volume", "frames")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
FQN = f"{CATALOG}.{SCHEMA}"

In [ ]:
import logging
import random
from datetime import datetime, timedelta, timezone

from pyspark.sql import Row

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("seed_data")

_RNG = random.Random(42)
_NOW = datetime.now(timezone.utc).replace(microsecond=0)

_STORE_DEFS: list[tuple[str, str, str, float, float]] = [
    ("S-ATL-001", "Store #1247 - Atlanta",        "Atlanta, GA",      33.7490, -84.3880),
    ("S-ATL-002", "Store #1248 - Atlanta North",   "Atlanta, GA",      33.9526, -84.5499),
    ("S-DAL-001", "Store #2145 - Dallas",          "Dallas, TX",       32.7767, -96.7970),
    ("S-HOU-001", "Store #2389 - Houston",         "Houston, TX",      29.7604, -95.3698),
    ("S-TAM-001", "Store #3421 - Tampa",           "Tampa, FL",        27.9506, -82.4572),
    ("S-TAM-002", "Store #3422 - Tampa Bay",       "Tampa, FL",        27.7676, -82.6403),
    ("S-NAS-001", "Store #4108 - Nashville",       "Nashville, TN",    36.1627, -86.7816),
    ("S-CHA-001", "Store #4215 - Charlotte",       "Charlotte, NC",    35.2271, -80.8431),
]

_LABELS = ["vehicle", "person", "truck", "package", "pizza"]
_STATES = ["GA", "FL", "TX", "AL", "SC", "NC", "TN", "MS"]
_STATE_WEIGHTS = [0.28, 0.24, 0.16, 0.11, 0.08, 0.06, 0.04, 0.03]

In [ ]:
# Assume the catalog already exists (most workspaces with Default Storage
# disallow ad-hoc CREATE CATALOG). We only ensure the schema + volumes.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQN}.{VOLUME}")
# `frames_inbox` is the drop point for the continuous detection pipeline:
# the simulator (or a Zerobus producer) writes raw images here, Auto Loader
# picks them up, and the pipeline dedupes + runs YOLO.
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQN}.frames_inbox")
LOG.info("Schema/volumes ready: %s.%s (volumes=%s, frames_inbox)", CATALOG, SCHEMA, VOLUME)

## Stores

In [ ]:
store_rows = [
    Row(id=sid, name=name, location=loc, lat=lat, lng=lng)
    for sid, name, loc, lat, lng in _STORE_DEFS
]
(spark.createDataFrame(store_rows)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.stores"))
LOG.info("Wrote %d stores", len(store_rows))

## Devices + readings

One temperature device per store, with a status drawn from a triangular
distribution around its current temperature. Readings are written hourly for
the last 7 days.

In [ ]:
def _classify(temp: float) -> str:
    if temp > 90.0:
        return "critical"
    if temp > 80.0:
        return "warning"
    return "normal"


def _build_devices() -> tuple[list[Row], list[Row]]:
    device_rows: list[Row] = []
    reading_rows: list[Row] = []
    for sid, name, loc, _lat, _lng in _STORE_DEFS:
        device_id = sid.replace("S-", "RT-") + "-T1"
        base = _RNG.uniform(70.0, 90.0)
        current = round(base + _RNG.uniform(-2.5, 5.0), 1)
        device_rows.append(Row(
            id=device_id,
            name=name,
            location=loc,
            current_temp=current,
            status=_classify(current),
            last_update=_NOW - timedelta(seconds=_RNG.randint(30, 600)),
        ))
        for hours_ago in range(0, 24 * 7):
            ts = _NOW - timedelta(hours=hours_ago)
            wave = 8.0 * (1.0 + (0.5 * (hours_ago % 24) / 24.0))
            temp = round(base + wave * (_RNG.random() - 0.5), 1)
            humidity = round(45.0 + _RNG.uniform(0, 25), 1)
            reading_rows.append(Row(
                device_id=device_id,
                ts=ts,
                temperature=temp,
                humidity=humidity,
                status=_classify(temp),
            ))
    return device_rows, reading_rows


_devices, _readings = _build_devices()
(spark.createDataFrame(_devices)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.devices"))
(spark.createDataFrame(_readings)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.device_readings"))
LOG.info("Wrote %d devices, %d readings", len(_devices), len(_readings))

## Camera status

Per-hour online flag for ~20 cameras across all stores. Used for the
"Online Cameras Frequency" chart on the overview.

In [ ]:
def _build_camera_status() -> list[Row]:
    cameras_per_store = 3
    rows: list[Row] = []
    for sid, *_ in _STORE_DEFS:
        for cam in range(cameras_per_store):
            cam_id = f"{sid}-CAM-{cam+1:02d}"
            for hours_ago in range(0, 24):
                ts = _NOW - timedelta(hours=hours_ago)
                online = _RNG.random() > 0.04
                rows.append(Row(camera_id=cam_id, store_id=sid, ts=ts, online=online))
    return rows


_cameras = _build_camera_status()
(spark.createDataFrame(_cameras)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.camera_status"))
LOG.info("Wrote %d camera_status rows", len(_cameras))

## Detections

Synthetic bounding-box detections for the last 30 days across all stores.
Class distribution leans heavily on `vehicle`, `person`, and `pizza` to match
the quick-serve restaurant scenario.

In [ ]:
_LABEL_WEIGHTS = {"vehicle": 0.42, "person": 0.28, "truck": 0.10, "package": 0.07, "pizza": 0.13}
_CLASS_IDS = {"vehicle": 2, "person": 0, "truck": 7, "package": 84, "pizza": 53}


def _build_detections() -> list[Row]:
    rows: list[Row] = []
    det_id = 0
    for days_ago in range(0, 30):
        per_day = _RNG.randint(220, 360)
        for _ in range(per_day):
            det_id += 1
            sid, *_rest = _RNG.choice(_STORE_DEFS)
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            label = _RNG.choices(list(_LABEL_WEIGHTS), weights=list(_LABEL_WEIGHTS.values()), k=1)[0]
            confidence = round(0.75 + _RNG.random() * 0.24, 3)
            x1 = _RNG.randint(20, 900)
            y1 = _RNG.randint(20, 500)
            x2 = x1 + _RNG.randint(40, 300)
            y2 = y1 + _RNG.randint(40, 220)
            frame_id = f"frame_{det_id:06d}"
            rows.append(Row(
                id=det_id,
                frame_id=frame_id,
                ts=ts,
                store_id=sid,
                label=label,
                class_id=_CLASS_IDS[label],
                confidence=confidence,
                bbox=[x1, y1, x2, y2],
            ))
    return rows


_dets = _build_detections()
(spark.createDataFrame(_dets)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.detections"))
LOG.info("Wrote %d detections", len(_dets))

## License plates

Plate captures over the last 30 days. Plate numbers are partially masked.

In [ ]:
def _build_plates() -> list[Row]:
    rows: list[Row] = []
    plate_id = 0
    for days_ago in range(0, 30):
        per_day = _RNG.randint(80, 160)
        for _ in range(per_day):
            plate_id += 1
            sid, *_rest = _RNG.choice(_STORE_DEFS)
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            state = _RNG.choices(_STATES, weights=_STATE_WEIGHTS, k=1)[0]
            confidence = round(0.90 + _RNG.random() * 0.09, 3)
            prefix = "".join(_RNG.choice("ABCDEFGHJKLMNPRSTUVWXYZ") for _ in range(3))
            rows.append(Row(
                id=plate_id,
                ts=ts,
                store_id=sid,
                state=state,
                plate_masked=f"{prefix}***",
                confidence=confidence,
            ))
    return rows


_plates = _build_plates()
(spark.createDataFrame(_plates)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.license_plates"))
LOG.info("Wrote %d license_plates", len(_plates))

## Inventory

Pizza stock percentage and truck-parking capacity for the first four stores
over the last 12 hours, sampled every 30 minutes.

In [ ]:
import math


def _build_inventory() -> list[Row]:
    rows: list[Row] = []
    for sid, *_ in _STORE_DEFS[:4]:
        for half_hours_ago in range(0, 24):
            ts = _NOW - timedelta(minutes=30 * half_hours_ago)
            hour_pos = (ts.hour + ts.minute / 60.0)
            pizza_pct = 100.0 - max(0.0, 12.0 * math.sin((hour_pos - 6) * math.pi / 12.0)) - _RNG.uniform(0, 8)
            pizza_pct = max(15.0, min(100.0, pizza_pct))
            truck_pct = 30.0 + 30.0 * math.sin((hour_pos - 6) * math.pi / 12.0) + _RNG.uniform(-6, 6)
            truck_pct = max(0.0, min(100.0, truck_pct))
            rows.append(Row(ts=ts, store_id=sid, item="pizza",         percentage=round(pizza_pct, 1)))
            rows.append(Row(ts=ts, store_id=sid, item="truck_parking", percentage=round(truck_pct, 1)))
    return rows


_inv = _build_inventory()
(spark.createDataFrame(_inv)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.inventory"))
LOG.info("Wrote %d inventory rows", len(_inv))

## Alerts (Jolt rules)

Recent rule-engine alert events for the Alerts tab.

In [ ]:
_ALERT_RULES = [
    ("temperature_critical", "Refrigeration temperature > 90F", "critical"),
    ("temperature_warning",  "Refrigeration temperature > 80F", "warning"),
    ("pizza_low_stock",      "Pizza inventory dropped below 25%", "warning"),
    ("camera_offline",       "Camera offline > 5 minutes", "warning"),
    ("vehicle_dwell_long",   "Vehicle dwell time > 8 minutes at drive-through", "info"),
    ("unrecognized_plate",   "Unrecognized license plate at restricted lane", "info"),
]


def _build_alerts() -> list[Row]:
    rows: list[Row] = []
    alert_id = 0
    for days_ago in range(0, 7):
        per_day = _RNG.randint(8, 18)
        for _ in range(per_day):
            alert_id += 1
            sid, store_name, *_ = _RNG.choice(_STORE_DEFS)
            rule_id, message, severity = _RNG.choice(_ALERT_RULES)
            seconds = _RNG.randint(0, 86_399)
            ts = _NOW - timedelta(days=days_ago, seconds=seconds)
            rows.append(Row(
                id=alert_id,
                ts=ts,
                store_id=sid,
                store_name=store_name,
                rule_id=rule_id,
                message=message,
                severity=severity,
                acknowledged=days_ago >= 1,
            ))
    return rows


_alerts = _build_alerts()
(spark.createDataFrame(_alerts)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{FQN}.alerts"))
LOG.info("Wrote %d alerts", len(_alerts))

In [ ]:
tables = ["stores", "devices", "device_readings", "camera_status", "detections", "license_plates", "inventory", "alerts"]
for tbl in tables:
    cnt = spark.table(f"{FQN}.{tbl}").count()
    LOG.info("%s.%s  rows=%d", FQN, tbl, cnt)